In [0]:
%sql
use catalog misgauravcatalog;
create database if not exists retaildb;
use retaildb; 

In [0]:
%sql
create table if not exists retaildb.orders_bronze
(
order_id int,
order_date string,
customer_id int,
order_status string,
filename string,
createdon timestamp
)
using delta
location "gs://databricksfolder/bronze/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists retaildb.orders_silver
(
order_id int,
order_date date,
customer_id int,
order_status string,
order_year int GENERATED ALWAYS AS (YEAR(order_date)),
order_month int GENERATED ALWAYS AS (MONTH(order_date)),
createdon timestamp,
modifiedon timestamp
)
using delta
location "gs://databricksfolder/silver/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists retaildb.orders_gold
(
customer_id int,
order_status string,
order_year int,
num_orders int
)
using delta
location "gs://databricksfolder/gold/"
TBLPROPERTIES(delta.enableChangeDataFeed = true)

In [0]:
%sql
copy into retaildb.orders_bronze from (
  select order_id::int,
  order_date::string,
  customer_id::int,
  order_status::string,
  _metadata.file_path as filename,
  current_timestamp() as createdon
  from 'gs://databricksfolder/raw/'
)
fileformat = CSV
format_options('header'='true');


In [0]:
%sql
create or replace temporary view orders_bronze_changes
AS 
select * from table_changes('retaildb.orders_bronze', 0) where order_id > 0
AND customer_id > 0 and order_status IN ("CLOSED", "PENDING_PAYMENT");

In [0]:
import os

# 1. Check if the physical _delta_log directory exists on GCS before doing ANY data operations
try:
    # We check if we can describe the history. If there's no transaction log, this built-in command fails cleanly.
    spark.sql("DESCRIBE HISTORY retaildb.orders_silver")
    table_initialized = True
except Exception:
    table_initialized = False

# 2. Run your pipeline routing logic based on the physical state
if not table_initialized:
    print("First run detected! Force-initializing the physical log folder on GCS...")
    
    # Run a dummy insert to establish the _delta_log directory safely
    spark.sql("""
        INSERT INTO retaildb.orders_silver (order_id, order_date, customer_id, order_status, createdon, modifiedon)
        VALUES (-1, CAST('1970-01-01' AS DATE), -1, 'INITIALIZED', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP())
    """)
    
    # Immediately clean it up
    spark.sql("DELETE FROM retaildb.orders_silver WHERE order_id = -1")
    print("Physical log structure initialized successfully!")


# 3. Now that the table physically exists on GCS, your MERGE will run perfectly every single time!
print("Running production MERGE upsert logic...")
spark.sql("""
    MERGE INTO retaildb.orders_silver tgt
    USING orders_bronze_changes src ON tgt.order_id = src.order_id
    WHEN MATCHED THEN
      UPDATE SET tgt.order_status = src.order_status, tgt.customer_id = src.customer_id, tgt.modifiedon = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN
      INSERT (order_id, order_date, customer_id, order_status, createdon, modifiedon) 
      VALUES (src.order_id, src.order_date, src.customer_id, src.order_status, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP())
""")
print("Silver layer pipeline processed successfully!")

In [0]:
%sql
insert overwrite table retaildb.orders_gold
select customer_id, order_status, order_year, count(order_id) as num_orders
from retaildb.orders_silver group by 1, 2, 3;
-- let's run this